# Multi-Model Classification — Training Notebook

Trains all 5 models required by `MODELS_SOURCE_OF_TRUTH.md` on the
**Breast Cancer Wisconsin (Diagnostic)** dataset (UCI ML Repository, bundled via
`sklearn.datasets.load_breast_cancer`), computes the 6 required metrics for each,
and saves all artifacts needed by `app.py`.

- Dataset: 569 instances, 30 numeric features, binary target (malignant / benign)
- Satisfies FR-1 (≥12 features, ≥500 rows) with margin to spare.
- Source: W.N. Street, W.H. Wolberg and O.L. Mangasarian, UCI Machine Learning
  Repository, https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)


In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.datasets import load_breast_cancer

pd.set_option("display.max_columns", None)
np.random.seed(42)


## Phase 1 — Data

Load, inspect, and confirm the dataset meets FR-1 (≥12 features, ≥500 rows).

In [2]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()
df = df.rename(columns={"target": "target"})

print("Shape:", df.shape)
print("Target classes:", raw.target_names.tolist())
print("Feature count (excluding target):", df.shape[1] - 1)
assert df.shape[0] >= 500, "Dataset must have at least 500 rows (FR-1)"
assert df.shape[1] - 1 >= 12, "Dataset must have at least 12 features (FR-1)"
df.head()


Shape: (569, 31)
Target classes: ['malignant', 'benign']
Feature count (excluding target): 30


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [3]:
# Null check (cleaning step per IMPLEMENTATION_GUIDE.md Part A, step 3)
print("Null values per column:")
print(df.isnull().sum().sum(), "total nulls")

# All features here are already numeric (no categorical encoding needed for this dataset)
df.dtypes.value_counts()


Null values per column:
0 total nulls


float64    30
int64       1
Name: count, dtype: int64

In [4]:
# Class balance check
df["target"].value_counts(normalize=True).rename({0: "malignant", 1: "benign"})


target
benign       0.627417
malignant    0.372583
Name: proportion, dtype: float64

## Train/test split

Fixed `random_state=42`, stratified, reused identically for every model (FR-2).


In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["target"])
y = df["target"]

FEATURE_NAMES = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (455, 30)
Test shape: (114, 30)


## Export `test_data.csv`

Held-out test split, written to the repo root, with the target column included so
the app's uploaded file has the same schema as training (FR-4 / TC-04).


In [6]:
test_export = X_test.copy()
test_export["target"] = y_test.values
test_export.to_csv("../test_data.csv", index=False)
print("Wrote ../test_data.csv with shape:", test_export.shape)
test_export.head()


Wrote ../test_data.csv with shape: (114, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
256,19.55,28.77,133.60,1207.0,0.09260,0.20630,0.17840,0.11440,0.1893,0.06232,0.8426,1.1990,7.158,106.400,0.006356,0.047650,0.038630,0.015190,0.01936,0.005252,25.05,36.27,178.60,1926.0,0.1281,0.53290,0.4251,0.19410,0.2818,0.10050,0
428,11.13,16.62,70.47,381.1,0.08151,0.03834,0.01369,0.01370,0.1511,0.06148,0.1415,0.9671,0.968,9.704,0.005883,0.006263,0.009398,0.006189,0.02009,0.002377,11.68,20.29,74.35,421.1,0.1030,0.06219,0.0458,0.04044,0.2383,0.07083,1
501,13.82,24.49,92.33,595.9,0.11620,0.16810,0.13570,0.06759,0.2275,0.07237,0.4751,1.5280,2.974,39.050,0.009680,0.038560,0.034760,0.016160,0.02434,0.006995,16.01,32.94,106.00,788.0,0.1794,0.39660,0.3381,0.15210,0.3651,0.11830,0
363,16.50,18.29,106.60,838.1,0.09686,0.08468,0.05862,0.04835,0.1495,0.05593,0.3389,1.4390,2.344,33.580,0.007257,0.018050,0.018320,0.010330,0.01694,0.002001,18.13,25.45,117.20,1009.0,0.1338,0.16790,0.1663,0.09123,0.2394,0.06469,1
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,1.1760,1.2560,7.673,158.700,0.010300,0.028910,0.051980,0.024540,0.01114,0.004239,25.45,26.40,166.10,2027.0,0.1410,0.21130,0.4107,0.22160,0.2060,0.07115,0


## Phase 2 — Modeling

Shared preprocessing pipeline (`StandardScaler`, required for the distance-based /
gradient-based models — Logistic Regression and kNN — and applied consistently to all
5 models for fair comparison per FR-2).


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")
joblib.dump(FEATURE_NAMES, "feature_names.pkl")
print("Saved scaler.pkl and feature_names.pkl")


Saved scaler.pkl and feature_names.pkl


### Train all 5 models (`MODELS_SOURCE_OF_TRUTH.md` §1)

`GaussianNB` is used (not `MultinomialNB`) because all 30 features are continuous
measurements, not counts/frequencies.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=100, random_state=42),
}

FILE_NAMES = {
    "Logistic Regression": "Logistic_Regression.pkl",
    "Decision Tree": "Decision_Tree.pkl",
    "kNN": "kNN.pkl",
    "Naive Bayes": "Naive_Bayes.pkl",
    "Random Forest (Ensemble)": "Random_Forest_Ensemble.pkl",
}


### Compute the 6 required metrics per model (`MODELS_SOURCE_OF_TRUTH.md` §2 / FR-3)

Accuracy, AUC, Precision, Recall, F1, MCC — using the sklearn functions specified in
the source-of-truth doc.


In [9]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

results = {}
train_accuracy = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)

    results[name] = {
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba[:, 1]), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
    }
    train_accuracy[name] = round(accuracy_score(y_train, model.predict(X_train_scaled)), 4)

    joblib.dump(model, FILE_NAMES[name])

with open("metrics.json", "w") as f:
    json.dump(results, f, indent=2)

metrics_df = pd.DataFrame(results).T
metrics_df


,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.9825,0.9954,0.9861,0.9861,0.9861,0.9623
Decision Tree,0.9123,0.9147,0.9559,0.9028,0.9286,0.8174
kNN,0.9561,0.9788,0.9589,0.9722,0.9655,0.9054
Naive Bayes,0.9298,0.9868,0.9444,0.9444,0.9444,0.8492
Random Forest (Ensemble),0.9561,0.9939,0.9589,0.9722,0.9655,0.9054


### Sanity checks (`MODELS_SOURCE_OF_TRUTH.md` §3)

- Naive Bayes / Logistic Regression AUC near 0.5 → check class balance/encoding.
- kNN underperforming → confirm scaling was applied.
- Decision Tree train acc ≈1.0 but test acc much lower → overfitting.
- Random Forest should match or beat the single Decision Tree.


In [10]:
print("Train accuracy per model (overfitting check):")
for name, acc in train_accuracy.items():
    print(f"  {name:30s} train_acc={acc:.4f}  test_acc={results[name]['Accuracy']:.4f}")

assert results["Random Forest (Ensemble)"]["Accuracy"] >= results["Decision Tree"]["Accuracy"] - 0.02, \
    "Random Forest should generally match or beat the single Decision Tree"
print("\nSanity checks passed.")


Train accuracy per model (overfitting check):
  Logistic Regression            train_acc=0.9890  test_acc=0.9825
  Decision Tree                  train_acc=0.9978  test_acc=0.9123
  kNN                            train_acc=0.9736  test_acc=0.9561
  Naive Bayes                    train_acc=0.9385  test_acc=0.9298
  Random Forest (Ensemble)       train_acc=1.0000  test_acc=0.9561

Sanity checks passed.


### Confusion matrix + classification report per model (used by `app.py` at inference time; shown here for the training-time view)

In [11]:
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    print(f"=== {name} ===")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=raw.target_names))
    print()


=== Logistic Regression ===
[[41  1]
 [ 1 71]]
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


=== Decision Tree ===
[[39  3]
 [ 7 65]]
              precision    recall  f1-score   support

   malignant       0.85      0.93      0.89        42
      benign       0.96      0.90      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.92      0.91       114
weighted avg       0.92      0.91      0.91       114


=== kNN ===
[[39  3]
 [ 2 70]]
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macr

## Comparison table (for README — matches `MODELS_SOURCE_OF_TRUTH.md` §4 template)

In [12]:
comparison = metrics_df.reset_index().rename(columns={"index": "ML Model Name"})
comparison = comparison[["ML Model Name", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]]
comparison.to_csv("comparison_table.csv", index=False)
comparison


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9861,0.9861,0.9861,0.9623
1,Decision Tree,0.9123,0.9147,0.9559,0.9028,0.9286,0.8174
2,kNN,0.9561,0.9788,0.9589,0.9722,0.9655,0.9054
3,Naive Bayes,0.9298,0.9868,0.9444,0.9444,0.9444,0.8492
4,Random Forest (Ensemble),0.9561,0.9939,0.9589,0.9722,0.9655,0.9054


## Saved artifacts

All files below are written into `model/` and are what `app.py` loads at inference
time — nothing here is recomputed by the app; it only reads these.


In [13]:
import os
print("Files in model/:")
for f in sorted(os.listdir(".")):
    if os.path.isfile(f):
        print(" -", f, f"({os.path.getsize(f)} bytes)")


Files in model/:
 - Decision_Tree.pkl (4105 bytes)
 - Logistic_Regression.pkl (1103 bytes)
 - Naive_Bayes.pkl (1735 bytes)
 - Random_Forest_Ensemble.pkl (324361 bytes)
 - comparison_table.csv (336 bytes)
 - feature_names.pkl (561 bytes)
 - kNN.pkl (113652 bytes)
 - metrics.json (767 bytes)
 - scaler.pkl (2103 bytes)
 - train_models.ipynb (50727 bytes)
